
# GVH Diagonal Cubic 0.3.2.7.3.1 — SSS Curvature and Full Dirac-DOF Correction Audit

**Auteur :** Charlemagne O Laurince  
**Version :** 0.3.2.7.3.1  
**Position :** correctif obligatoire de `0.3.2.7.3` avant toute dispersion \(\omega^2(k)\)

---

## Objectif

Ce notebook corrige exactement deux points identifiés dans l'audit de `0.3.2.7.3` :

1. **Snapshot SSS 3D invalide** : la métrique
   \[
   h_{ij}=\mathrm{diag}(1,1+r^2,1+r^2)
   \]
   dépendait de \(r\), mais le calcul de Christoffel/Ricci utilisait les coordonnées `[x,y,z]`. SymPy traitait donc \(r\) comme un paramètre indépendant et annulait artificiellement toutes les dérivées.

2. **Comptage DOF prématuré** : les valeurs \(2,3,3\) avaient été qualifiées de « Dirac DOF » sans analyse hamiltonienne complète du secteur GVH contenant notamment \(u^\mu\), le multiplicateur de norme et les contraintes secondaires potentielles.

**Règle de ce notebook :** aucun calcul de dispersion \(\omega^2(k)\) n'est autorisé tant que ces deux corrections ne sont pas enregistrées.


In [1]:

from __future__ import annotations

from pathlib import Path
import json
import sys
import sympy as sp
import pandas as pd

print("GVH Diagonal Cubic 0.3.2.7.3.1")
print("Python:", sys.version.split()[0])
print("SymPy:", sp.__version__)


GVH Diagonal Cubic 0.3.2.7.3.1
Python: 3.12.13
SymPy: 1.14.0



# 1. Correction du snapshot SSS 3D

La métrique jouet conservée depuis `0.3.2.7.3` est

\[
h_{ij}(r)=
\begin{pmatrix}
1&0&0\\
0&1+r^2&0\\
0&0&1+r^2
\end{pmatrix}.
\]

Pour cette métrique, \(r\) doit être **une coordonnée réelle du calcul différentiel**.

Nous choisissons donc

\[
x^i=(r,y,z)
\]

et non \((x,y,z)\) avec \(r\) externe.

Ce test ne prétend pas être une métrique sphérique standard ; il est seulement un snapshot 3D dépendant radialement, utilisé comme contrôle de la chaîne symbolique.


In [2]:

r, y, z = sp.symbols("r y z", real=True, positive=True)

h_sss = sp.diag(
    1,
    1 + r**2,
    1 + r**2,
)
coords_sss = [r, y, z]

def christoffel(g, coords):
    n = len(coords)
    gi = sp.simplify(g.inv())
    Gamma = [[[sp.S.Zero for _ in range(n)] for _ in range(n)] for _ in range(n)]

    for a in range(n):
        for b in range(n):
            for c in range(n):
                Gamma[a][b][c] = sp.simplify(
                    sum(
                        gi[a,d] * (
                            sp.diff(g[d,c], coords[b])
                            + sp.diff(g[d,b], coords[c])
                            - sp.diff(g[b,c], coords[d])
                        )
                        for d in range(n)
                    ) / 2
                )
    return gi, Gamma

def ricci_tensor_and_scalar(g, coords, Gamma):
    n = len(coords)
    gi = sp.simplify(g.inv())
    Ric = sp.zeros(n,n)

    for mu in range(n):
        for nu in range(n):
            expr = sp.S.Zero
            for rho in range(n):
                expr += sp.diff(Gamma[rho][mu][nu], coords[rho])
                expr -= sp.diff(Gamma[rho][mu][rho], coords[nu])

                for sig in range(n):
                    expr += Gamma[rho][rho][sig] * Gamma[sig][mu][nu]
                    expr -= Gamma[rho][nu][sig] * Gamma[sig][mu][rho]

            Ric[mu,nu] = sp.simplify(expr)

    R = sp.simplify(
        sum(
            gi[mu,nu] * Ric[mu,nu]
            for mu in range(n)
            for nu in range(n)
        )
    )
    return Ric, R

h_inv, Gamma_sss = christoffel(h_sss, coords_sss)
Ric_sss, R_sss = ricci_tensor_and_scalar(h_sss, coords_sss, Gamma_sss)

nonzero_gamma = [
    (a,b,c,sp.simplify(Gamma_sss[a][b][c]))
    for a in range(3)
    for b in range(3)
    for c in range(3)
    if sp.simplify(Gamma_sss[a][b][c]) != 0
]

print("Coordinates used:", coords_sss)
print("Non-zero Christoffel symbols:", len(nonzero_gamma))
for item in nonzero_gamma:
    print(item)

print("\nRicci tensor:")
sp.pprint(Ric_sss)

print("\n3D Ricci scalar:")
sp.pprint(R_sss)


Coordinates used: [r, y, z]
Non-zero Christoffel symbols: 6
(0, 1, 1, -r)
(0, 2, 2, -r)
(1, 0, 1, r/(r**2 + 1))
(1, 1, 0, r/(r**2 + 1))
(2, 0, 2, r/(r**2 + 1))
(2, 2, 0, r/(r**2 + 1))

Ricci tensor:
⎡     -2              ⎤
⎢─────────────  0   0 ⎥
⎢ 4      2            ⎥
⎢r  + 2⋅r  + 1        ⎥
⎢                     ⎥
⎢      0        -1  0 ⎥
⎢                     ⎥
⎣      0        0   -1⎦

3D Ricci scalar:
  ⎛   2    ⎞ 
2⋅⎝- r  - 2⎠ 
─────────────
 4      2    
r  + 2⋅r  + 1



## 1.1 Résultat corrigé

Le calcul précédent doit produire un nombre **non nul** de symboles de Christoffel et une courbure non artificiellement annulée.

Pour ce snapshot précis,

\[
h_{ij}=\mathrm{diag}(1,1+r^2,1+r^2),
\]

le scalaire de Ricci obtenu directement par la définition tensorielle est

\[
\boxed{
{}^{(3)}R
=
-\frac{2(r^2+2)}{(1+r^2)^2}
}.
\]

Cette valeur remplace le résultat nul artificiel de `0.3.2.7.3`.

Elle remplace également toute valeur antérieure attribuée à ce snapshot tant qu'une autre métrique 3D explicitement définie n'est pas utilisée.


In [3]:

R_expected = sp.simplify(
    -2*(r**2 + 2)/(1+r**2)**2
)

assert len(nonzero_gamma) > 0
assert sp.simplify(R_sss - R_expected) == 0

print("PASS — corrected SSS-coordinate audit.")
print("^(3)R =", sp.factor(R_sss))


PASS — corrected SSS-coordinate audit.
^(3)R = -2*(r**2 + 2)/(r**2 + 1)**2



# 2. Correction du gate G3 — degrés de liberté

Le notebook précédent utilisait des expressions de type

\[
10-2\times4=2
\]

ou

\[
6-3=3.
\]

Ce sont au mieux des **comptages heuristiques de variables de configuration**. Ils ne constituent pas une analyse complète de Dirac.

Pour un système contraint, le comptage correct se fait dans l'espace des phases :

\[
\boxed{
N_{\mathrm{DOF}}
=
\frac{
N_{\mathrm{phase}}
-2N_{\mathrm{1st}}
-N_{\mathrm{2nd}}
}{2}
}.
\]

Pour le secteur DC-4 GVH candidat, il faut notamment tenir compte de :

- \(g_{\mu\nu}\) ou des variables ADM \((h_{ij},N,N^i)\),
- \(u^\mu\),
- du multiplicateur imposant \(u^\mu u_\mu=-1\),
- des moments canoniques correspondants,
- des contraintes primaires,
- des contraintes secondaires,
- de leur classification première/seconde classe,
- et de la dépendance éventuelle du résultat aux combinaisons \(c_i\).

Tant que cette chaîne n'est pas explicitement calculée, **G3 ne peut pas être PASS**.


In [4]:

def dirac_dof(N_phase, N_first, N_second):
    return sp.Rational(N_phase - 2*N_first - N_second, 2)

# Benchmark uniquement : GR pure en ADM.
# 6 h_ij + 6 pi^ij = 12 variables de phase,
# 4 contraintes de première classe (H + 3 H_i).
pure_GR_benchmark = dirac_dof(
    N_phase=12,
    N_first=4,
    N_second=0,
)

assert pure_GR_benchmark == 2

print("Pure-GR ADM benchmark =", pure_GR_benchmark)
print("IMPORTANT: this is NOT assigned to the GVH DC-4 vector-tensor model.")


Pure-GR ADM benchmark = 2
IMPORTANT: this is NOT assigned to the GVH DC-4 vector-tensor model.



## 2.1 Registre canonique requis

Au lieu d'inventer des DOF numériques, nous enregistrons les objets qui doivent être dérivés pour chaque architecture.


In [5]:

canonical_registry = pd.DataFrame([
    {
        "architecture": "DC-4 GVH candidate",
        "configuration_fields": "ADM metric variables + u^mu + norm multiplier",
        "phase_space_dimension": "TO_DERIVE",
        "primary_constraints": "TO_DERIVE",
        "secondary_constraints": "TO_DERIVE",
        "first_class_constraints": "TO_DERIVE",
        "second_class_constraints": "TO_DERIVE",
        "physical_DOF": "OPEN",
    },
    {
        "architecture": "DC-3+Flow",
        "configuration_fields": "h_ij(x,lambda) plus whatever lapse/shift/auxiliary fields explicit action requires",
        "phase_space_dimension": "TO_DERIVE",
        "primary_constraints": "TO_DERIVE_FROM_ACTION",
        "secondary_constraints": "TO_DERIVE_FROM_ACTION",
        "first_class_constraints": "TO_DERIVE",
        "second_class_constraints": "TO_DERIVE",
        "physical_DOF": "OPEN",
    },
    {
        "architecture": "DC-3+Emergent Clock",
        "configuration_fields": "DC-3+Flow variables plus tau only if introduced as independent auxiliary variable",
        "phase_space_dimension": "TO_DERIVE",
        "primary_constraints": "TO_DERIVE",
        "secondary_constraints": "TO_DERIVE including tau-F[h] consistency",
        "first_class_constraints": "TO_DERIVE",
        "second_class_constraints": "TO_DERIVE",
        "physical_DOF": "OPEN / Flag-H1 separate",
    },
])

canonical_registry


,architecture,configuration_fields,phase_space_dimension,primary_constraints,secondary_constraints,first_class_constraints,second_class_constraints,physical_DOF
0,DC-4 GVH candidate,ADM metric variables + u^mu + norm multiplier,TO_DERIVE,TO_DERIVE,TO_DERIVE,TO_DERIVE,TO_DERIVE,OPEN
1,DC-3+Flow,"h_ij(x,lambda) plus whatever lapse/shift/auxil...",TO_DERIVE,TO_DERIVE_FROM_ACTION,TO_DERIVE_FROM_ACTION,TO_DERIVE,TO_DERIVE,OPEN
2,DC-3+Emergent Clock,DC-3+Flow variables plus tau only if introduce...,TO_DERIVE,TO_DERIVE,TO_DERIVE including tau-F[h] consistency,TO_DERIVE,TO_DERIVE,OPEN / Flag-H1 separate



# 3. Gate table corrigée

Après ce correctif :

| Gate | Statut |
|---|---|
| G0 — définitions DC-3+Flow / DC-4 | PASS |
| G1 — égalité des actions | OPEN |
| G2 — égalité des EOM | OPEN |
| G3 — DOF Dirac | **OPEN — FULL HAMILTONIAN/DIRAC ANALYSIS REQUIRED** |
| G4 — reparamétrisation \(\lambda\) | OPEN — dépend de l'action |
| G5 — snapshot SSS 3D | **PASS après correction des coordonnées** |
| Flag-H1 — monotonie de l'horloge émergente | UNPROVEN |

Le verdict architectural global reste donc

\[
\boxed{\text{ARCHITECTURE NOT YET UNIQUELY DETERMINED}}.
\]


In [6]:

GATES_0731 = {
    "G0_architecture_definitions": "PASS",
    "G1_reduced_action": "OPEN",
    "G2_EOM_equality": "OPEN",
    "G3_full_Dirac_DOF": "OPEN_FULL_HAMILTONIAN_DIRAC_ANALYSIS_REQUIRED",
    "G4_lambda_reparametrisation": "OPEN_ACTION_DEPENDENT",
    "G5_corrected_SSS_3curvature": "PASS",
    "FLAG_H1_emergent_clock_monotonicity": "UNPROVEN",
}

for k,v in GATES_0731.items():
    print(f"{k}: {v}")

assert GATES_0731["G3_full_Dirac_DOF"].startswith("OPEN")
assert GATES_0731["G5_corrected_SSS_3curvature"] == "PASS"


G0_architecture_definitions: PASS
G1_reduced_action: OPEN
G2_EOM_equality: OPEN
G3_full_Dirac_DOF: OPEN_FULL_HAMILTONIAN_DIRAC_ANALYSIS_REQUIRED
G4_lambda_reparametrisation: OPEN_ACTION_DEPENDENT
G5_corrected_SSS_3curvature: PASS
FLAG_H1_emergent_clock_monotonicity: UNPROVEN



# 4. Interdiction de lancer \(\omega^2(k)\) prématurément

Les relations de dispersion doivent être calculées **après** identification des modes physiques.

Sinon un mode contraint, auxiliaire ou purement de jauge pourrait être traité comme une excitation propagative.

Le gate de lancement est donc :

\[
\boxed{
\text{DISPERSION\_READY}
=
(G1\ \text{fermé})
\land
(G3\ \text{fermé})
\land
(\text{secteur cinétique cohérent})
}
\]

À la fin de `0.3.2.7.3.1`, ce gate doit rester `False`.


In [7]:

DISPERSION_READY = False

blocking_reasons = [
    "G1: explicit DC-3+Flow action not yet closed",
    "G3: full Hamiltonian/Dirac constraint classification not yet derived",
]

assert DISPERSION_READY is False

print("DISPERSION_READY =", DISPERSION_READY)
for reason in blocking_reasons:
    print("BLOCK:", reason)


DISPERSION_READY = False
BLOCK: G1: explicit DC-3+Flow action not yet closed
BLOCK: G3: full Hamiltonian/Dirac constraint classification not yet derived



# 5. Artefact machine-readable

Ce JSON remplace, pour les deux points corrigés ici, les valeurs `G3=PASS` et le snapshot SSS erroné du notebook 0.3.2.7.3.


In [8]:

artifact = {
    "notebook": "GVH_Diagonal_Cubic_0.3.2.7.3.1",
    "purpose": "Correct SSS coordinate/Ricci audit and retract premature physical-DOF counts",
    "corrections": {
        "SSS_snapshot": {
            "metric": "diag(1,1+r^2,1+r^2)",
            "coordinates": ["r","y","z"],
            "nonzero_Christoffel_count": len(nonzero_gamma),
            "Ricci_scalar": str(sp.factor(R_sss)),
            "status": "PASS",
        },
        "Dirac_DOF": {
            "previous_numeric_assignments_2_3_3": "RETRACTED_AS_UNPROVEN_FOR_GVH_ARCHITECTURES",
            "pure_GR_2_DOF": "BENCHMARK_ONLY",
            "GVH_DC4_physical_DOF": "OPEN",
            "DC3_Flow_physical_DOF": "OPEN",
            "DC3_Emergent_physical_DOF": "OPEN",
            "required_formula": "(N_phase - 2*N_first - N_second)/2",
            "status": "OPEN_FULL_HAMILTONIAN_DIRAC_ANALYSIS_REQUIRED",
        },
    },
    "gates": GATES_0731,
    "dispersion_ready": DISPERSION_READY,
    "global_architecture_verdict": "ARCHITECTURE_NOT_YET_UNIQUELY_DETERMINED",
    "next_step": "derive explicit Hamiltonian/Dirac constraint structure before omega^2(k)",
}

if Path("/content").exists():
    export_dir = Path("/content/gvh_exports")
else:
    export_dir = Path.cwd() / "gvh_exports"

export_dir.mkdir(parents=True, exist_ok=True)
artifact_path = export_dir / "gvh_0.3.2.7.3.1_correction_verdict.json"
artifact_path.write_text(json.dumps(artifact, indent=2, ensure_ascii=False), encoding="utf-8")

assert artifact_path.exists()
assert artifact["dispersion_ready"] is False

print("Artifact:", artifact_path)


Artifact: /content/gvh_exports/gvh_0.3.2.7.3.1_correction_verdict.json



# Conclusion

`0.3.2.7.3.1` effectue les deux corrections nécessaires avant toute analyse de dispersion :

1. le snapshot 3D utilise désormais réellement \(r\) comme coordonnée et donne
   \[
   \boxed{{}^{(3)}R=-\frac{2(r^2+2)}{(1+r^2)^2}};
   \]

2. les nombres \(2,3,3\) ne sont plus présentés comme DOF physiques des architectures GVH.  
   **G3 est rouvert** jusqu'à une analyse hamiltonienne/Dirac complète.

Le verdict architectural reste

\[
\boxed{\text{ARCHITECTURE NOT YET UNIQUELY DETERMINED}}
\]

et

\[
\boxed{\text{DISPERSION\_READY=False}}.
\]

La prochaine étape doit fermer le comptage canonique et l'action explicite avant de calculer \(\omega^2(k)\).
